# Inverse-square law: rate vs source distance

This notebook takes the `scintillator_distance_*.li` series (one 10-s file per source position) and tests whether the measured event rate falls off with distance as expected for a point source.

**Model.** For a point source of activity $A_0$ in front of a small detector, the rate observed is
$$
r(d) \;=\; \frac{A}{(d + d_0)^2} \;+\; B,
$$
with three free parameters:
- $A$: amplitude (absorbs source activity, detection efficiency, detector area).
- $d_0$: geometric offset between the holder notch labelled `0` and the scintillator center. The notch is *not* at the scintillator face, so the actual source–detector distance is $d + d_0$.
- $B$: distance-independent background (dark counts, cosmics, room background). Pulses from these sources do not care how far the $^{137}$Cs source is, so they show up as a constant floor.

If you do not include $d_0$ and $B$ in the fit, a literal $1/d^2$ curve will fail — not because the physics is wrong, but because you are fitting the wrong model.

In [ ]:
import glob
import re

import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as optimize

import moku_util as mu

## 1. Locate the data files

Glob the per-distance files and sort by the integer distance encoded in the filename. We work with `.li` paths and let `mu.DataFile` handle the `.npy` conversion (it caches the result, so the conversion runs at most once per file).

In [ ]:
data_dir = '../data'
files = sorted(
    glob.glob(f'{data_dir}/scintillator_distance_*.li'),
    key=lambda f: int(re.search(r'distance_(\d+)_', f).group(1)),
)
distances_cm = [int(re.search(r'distance_(\d+)_', f).group(1)) for f in files]
print(f'Found {len(files)} files, distances (cm): {distances_cm}')

## 2. Pick channel and peak-finding settings

**Channel.** Input 2 (`channel=1`) is the positive-going slow channel from the SiPM readout. Input 1 is inverted; if you want to use it instead, pass `negative_signal=True` to `find_peaks`.

**Prominence vs. height.** Input 2 has a DC offset of about $+0.1$ V, so `height` (an *absolute* threshold) is sensitive to that offset. `prominence` measures rise above the local baseline, which is what we actually mean by "a pulse". For a distance scan that is reproducibly compared across files, prominence is the safer choice.

**Threshold value.** We start with `prominence = 0.20 V`. We sweep it later as a robustness check.

In [ ]:
CHANNEL = 1            # Input 2 (slow, positive pulses)
PROMINENCE = 0.20      # V, threshold for an event to count
MIN_WIDTH = 2          # samples; rejects single-sample spikes

## 3. Extract a count rate per file

For each file: find peaks on the chosen channel, count them, and record $(d, N, T, r=N/T, \sigma_r = \sqrt{N}/T)$. The $\sqrt{N}$ uncertainty assumes the counts in a fixed time window are Poisson-distributed, which is what the time-difference fit in `time_analysis.ipynb` is supposed to verify on a single file.

In [ ]:
def rates_for_files(files, prominence, channel=1, width=2):
    """Return arrays (d, N, T, rate, sigma) for the given file list."""
    d_list, N_list, T_list = [], [], []
    for f in files:
        d_cm = int(re.search(r'distance_(\d+)_', f).group(1))
        df = mu.DataFile(f, verbose=False)
        df.find_peaks(
            channel=channel,
            height=None,
            width=width,
            prominence=(prominence, None),
        )
        N = len(df.peaks[channel])
        T = df.time[-1] - df.time[0]
        d_list.append(d_cm)
        N_list.append(N)
        T_list.append(T)

    d = np.array(d_list, dtype=float)
    N = np.array(N_list, dtype=float)
    T = np.array(T_list, dtype=float)
    rate = N / T
    sigma = np.sqrt(np.maximum(N, 1.0)) / T  # avoid /0 if a file had zero counts
    return d, N, T, rate, sigma

d, N, T, rate, sigma = rates_for_files(
    files, prominence=PROMINENCE, channel=CHANNEL, width=MIN_WIDTH
)

for di, Ni, Ti, ri, si in zip(d, N, T, rate, sigma):
    print(f'  d = {di:5.1f} cm   N = {int(Ni):6d}   T = {Ti:5.2f} s   rate = {ri:7.2f} +/- {si:.2f} Hz')

## 4. Exclude `distance_0`

The `distance_0` file was acquired ~12 minutes before the rest of the scan (check the timestamps in the filenames), so it was almost certainly a setup/test run rather than part of the systematic series. Its rate is also wildly inconsistent with the other points. Drop it from the fit but keep the rest.

In [ ]:
keep = d > 0
d_fit, N_fit, T_fit, rate_fit, sigma_fit = d[keep], N[keep], T[keep], rate[keep], sigma[keep]
print(f'Using {keep.sum()} points: d = {d_fit.astype(int).tolist()} cm')

## 5. Fit the inverse-square model

$$r(d) = \frac{A}{(d + d_0)^2} + B$$

We pass the Poisson uncertainties as `sigma=` and set `absolute_sigma=True` so the returned covariance is meaningful (i.e. the parameter errors are not rescaled to force $\chi^2/\text{dof} = 1$).

In [ ]:
def inverse_square(d, A, d0, B):
    return A / (d + d0) ** 2 + B

# Rough initial guesses: take the highest-rate point and assume B is comparable to the smallest rate.
p0 = [rate_fit[0] * (d_fit[0] + 1.0) ** 2, 1.0, rate_fit[-1] / 2.0]

popt, pcov = optimize.curve_fit(
    inverse_square,
    d_fit,
    rate_fit,
    sigma=sigma_fit,
    absolute_sigma=True,
    p0=p0,
    bounds=([0.0, 0.0, 0.0], [np.inf, 50.0, np.inf]),
)
A_fit, d0_fit, B_fit = popt
A_err, d0_err, B_err = np.sqrt(np.diag(pcov))

resid = rate_fit - inverse_square(d_fit, *popt)
chi2 = np.sum((resid / sigma_fit) ** 2)
ndof = len(d_fit) - len(popt)

print(f'  A   = {A_fit:8.1f} +/- {A_err:.1f}   (Hz * cm^2)')
print(f'  d0  = {d0_fit:8.2f} +/- {d0_err:.2f} cm')
print(f'  B   = {B_fit:8.2f} +/- {B_err:.2f} Hz')
print(f'  reduced chi^2 = {chi2 / ndof:.2f}   ({ndof} dof)')

## 6. Plot the data with the fit and residuals

Two panels:
- Top: rate vs distance with error bars and the fitted curve, on log-y so the inverse-square shape is visible.
- Bottom: pull residuals $(r_i - \text{model})/\sigma_i$. If the model is correct, these should scatter around zero with a spread of about 1 and no obvious shape.

In [ ]:
d_plot = np.linspace(d_fit.min() - 0.5, d_fit.max() + 0.5, 400)
model_plot = inverse_square(d_plot, *popt)

fig, ax = plt.subplots(
    2, 1, figsize=(6.5, 6), height_ratios=[3, 1], sharex=True
)

ax[0].errorbar(d_fit, rate_fit, yerr=sigma_fit, fmt='o', capsize=3, label='data')
ax[0].plot(d_plot, model_plot, 'r--', label=r'$A/(d+d_0)^2 + B$')
ax[0].axhline(B_fit, color='gray', ls=':', label=f'background $B$ = {B_fit:.1f} Hz')
ax[0].set_ylabel('Rate (Hz)')
ax[0].set_yscale('log')
ax[0].legend()

pull = (rate_fit - inverse_square(d_fit, *popt)) / sigma_fit
ax[1].scatter(d_fit, pull)
ax[1].axhline(0, color='k', ls='--')
ax[1].axhline(+1, color='gray', ls=':')
ax[1].axhline(-1, color='gray', ls=':')
ax[1].set_xlabel('Notch distance $d$ (cm)')
ax[1].set_ylabel(r'pull $(r-\hat r)/\sigma$')

fig.suptitle(
    f'prominence = {PROMINENCE} V    '
    f'$d_0$ = {d0_fit:.2f} +/- {d0_err:.2f} cm    '
    f'$\\chi^2$/dof = {chi2/ndof:.2f}'
)
fig.tight_layout()
plt.show()

## 7. Threshold robustness check

If the inverse-square shape is real physics, then changing the prominence threshold should change $A$ (different detection efficiency) and $B$ (different dark-count rate) but **not** the fitted $d_0$ or the goodness of the shape. If raising the threshold changes the *shape*, the low-threshold data is contaminated by something distance-independent (dark counts) or the model is wrong.

Refit at several thresholds and compare.

In [ ]:
thresholds = [0.15, 0.20, 0.25, 0.30]

results = []
for h in thresholds:
    d_h, N_h, T_h, r_h, s_h = rates_for_files(
        files, prominence=h, channel=CHANNEL, width=MIN_WIDTH
    )
    keep = d_h > 0
    d_h, r_h, s_h = d_h[keep], r_h[keep], s_h[keep]
    p0 = [r_h[0] * (d_h[0] + 1.0) ** 2, 1.0, r_h[-1] / 2.0]
    popt_h, pcov_h = optimize.curve_fit(
        inverse_square, d_h, r_h,
        sigma=s_h, absolute_sigma=True, p0=p0,
        bounds=([0.0, 0.0, 0.0], [np.inf, 50.0, np.inf]),
    )
    perr_h = np.sqrt(np.diag(pcov_h))
    resid = r_h - inverse_square(d_h, *popt_h)
    chi2 = np.sum((resid / s_h) ** 2)
    ndof = len(d_h) - len(popt_h)
    results.append((h, d_h, r_h, s_h, popt_h, perr_h, chi2 / ndof))

print(f'{"threshold (V)":>14}  {"A":>10}  {"d0 (cm)":>10}  {"B (Hz)":>10}  {"chi2/dof":>10}')
for h, _, _, _, popt_h, perr_h, rchi2 in results:
    print(
        f'{h:14.2f}  '
        f'{popt_h[0]:6.0f}+/-{perr_h[0]:4.0f}  '
        f'{popt_h[1]:5.2f}+/-{perr_h[1]:4.2f}  '
        f'{popt_h[2]:5.1f}+/-{perr_h[2]:4.1f}  '
        f'{rchi2:10.2f}'
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for h, d_h, r_h, s_h, popt_h, _, _ in results:
    line = ax.errorbar(d_h, r_h, yerr=s_h, fmt='o', capsize=2, label=f'h = {h:.2f} V')
    d_plot = np.linspace(d_h.min() - 0.5, d_h.max() + 0.5, 400)
    ax.plot(d_plot, inverse_square(d_plot, *popt_h), '--', color=line[0].get_color())
ax.set_xlabel('Notch distance $d$ (cm)')
ax.set_ylabel('Rate (Hz)')
ax.set_yscale('log')
ax.legend()
fig.tight_layout()
plt.show()

## 8. Interpreting the threshold sweep

Read the table from the previous cell:
- $A$ should *decrease* with threshold (you reject more real signal pulses) but the *shape* of the curve should stay the same.
- $B$ should *decrease* with threshold (you reject more dark counts).
- $d_0$ should be roughly **stable across thresholds** — it is a geometric property of the apparatus, not of the cut.
- Reduced $\chi^2$ rises with threshold. This is not a sign that the fit gets worse in any physical sense; it means that as the Poisson error bars shrink, you start to *resolve* small systematic deviations from the simple point-source $1/(d+d_0)^2$ model (finite scintillator face, scattering in the housing, slight off-axis geometry). At low threshold those deviations are hidden inside the larger error bars.

A reduced $\chi^2$ *well below* 1 (as at the lowest threshold) is itself a warning sign: it usually means the model has too much freedom relative to the information content. Here, when $B$ is comparable to the largest signal rate, the fit becomes nearly a flat line with a tiny inverse-square correction, and almost any reasonable $A$, $d_0$ can be absorbed into the constant. The fact that the parameter uncertainties are *also* much larger at the lowest threshold confirms this.

## 9. (Optional) Compare to the source activity

From the fitted $A$ at your chosen threshold, the predicted rate at distance $r = d + d_0$ from a true point source of activity $A_0$ would be
$$
r(d) = A_0 \cdot \frac{\Omega(r)}{4\pi} \cdot \varepsilon,
$$
where $\Omega/(4\pi) \approx \pi R^2 / (4\pi r^2)$ for a small detector of radius $R$, and $\varepsilon$ lumps together photopeak efficiency in NaI, attenuation in the source housing, and any threshold-dependent loss. Equating with the fit:
$$
A \;\approx\; A_0 \cdot \frac{R^2}{4} \cdot \varepsilon
\quad\Longrightarrow\quad
\varepsilon \;\approx\; \frac{4 A}{A_0 R^2}.
$$
Plug in $A_0$ from the source label (in decays/s, after correcting for the time since calibration) and the scintillator radius $R$. An $\varepsilon$ of a few percent at the 0.20 V threshold is the right order of magnitude for an unshielded NaI puck and a 662 keV gamma; an $\varepsilon$ near 1 or above 100% means something is wrong (wrong $R$, wrong $A_0$, threshold counting noise as signal, etc.).

For a more accurate model at small $d$, replace the small-detector approximation with the exact disk solid angle
$$
\frac{\Omega(r)}{4\pi} = \tfrac{1}{2}\left(1 - \frac{r}{\sqrt{r^2 + R^2}}\right),
$$
and fit `rate(d) = A0 * eff * 0.5 * (1 - (d+d0)/sqrt((d+d0)**2 + R**2)) + B` with $R$ either fixed (from a ruler) or floated. Deviations from the simple $1/r^2$ at small $d$ should be visible in the residuals from §6 and partially absorbed by the disk formula.